# Módulo 2 — Detección no supervisada de anomalías (fraude zero-day)

El **Módulo 1** (`src/models/train.py`) entrena modelos supervisados con ejemplos de fraude ya etiquetados — funcionan bien, pero solo pueden reconocer patrones parecidos a fraude que ya ocurrió antes. Un esquema de fraude genuinamente nuevo ("zero-day") no se parece a nada visto en el entrenamiento, y un modelo supervisado no tiene por qué detectarlo.

Este notebook explora el enfoque complementario: **aprender solo la forma de lo normal** (entrenando exclusivamente con transacciones legítimas) y marcar como anómalo cualquier caso que no se ajuste a ese patrón — sin necesidad de haber visto fraude antes.

Se comparan dos algoritmos de `scikit-learn`:
- **Isolation Forest**: aísla puntos con particiones aleatorias; las anomalías requieren menos particiones para quedar aisladas.
- **Local Outlier Factor (modo *novelty*)**: compara la densidad local de un punto contra la de sus vecinos más cercanos.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import RobustScaler

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.unsupervised.loader import get_unsupervised_data
from src.unsupervised.models import anomaly_score, build_isolation_forest, build_lof
from src.unsupervised.train_unsupervised import evaluate, plot_pr_curve, plot_score_distributions

COLOR_NOFRAUD = "#2a78d6"
COLOR_FRAUD = "#eb6834"


## 1. Datos: entrenamiento 100% normal, prueba mixta

`get_unsupervised_data()` reutiliza el dataset PaySim ya limpio (`clean_data` + `build_features`, las mismas funciones del Módulo 1) y separa:
- **Train**: una muestra de transacciones normales — el modelo nunca ve un fraude durante el ajuste.
- **Test**: una muestra de normales + *todas* las transacciones fraudulentas disponibles, para tener suficientes anomalías reales con las que medir desempeño.


In [ ]:
X_train, X_test, y_test = get_unsupervised_data()

print(f"Train (solo normales): {X_train.shape}")
print(f"Test (mixto): {X_test.shape} — fraude: {int(y_test.sum())} ({y_test.mean():.2%})")


## 2. Escalado

Local Outlier Factor depende de distancias euclidianas, así que las features deben estar en escalas comparables. Usamos `RobustScaler` (mediana / rango intercuartil) por la fuerte asimetría de `amount` y los saldos — es más resistente a valores extremos que un escalado estándar. El escalador se ajusta **solo con datos de entrenamiento** (normales), como corresponde en detección de anomalías: nunca debe ver la distribución de las anomalías de prueba.


In [ ]:
scaler = RobustScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


## 3. Entrenamiento de los dos modelos candidatos


In [ ]:
models = {
    "isolation_forest": build_isolation_forest(),
    "lof": build_lof(),
}

results = {}
for name, model in models.items():
    model.fit(X_train_scaled)
    scores = anomaly_score(model, X_test_scaled)
    metrics = evaluate(y_test, scores)
    results[name] = {"model": model, "scores": scores, **metrics}
    print(f"{name}: PR-AUC={metrics['pr_auc']:.4f}")


## 4. Precision@k y Recall@k

Métrica orientada a un flujo de trabajo realista: *"si un analista revisa las k transacciones más anómalas señaladas, ¿cuántas son fraude real?"* — más útil operativamente que accuracy o incluso ROC-AUC bajo un desbalance tan extremo.


In [ ]:
for name, res in results.items():
    print(f"\n{name}")
    for k, (precision, recall) in res["precision_recall_at_k"].items():
        print(f"  Precision@{k}: {precision:.4f} | Recall@{k}: {recall:.4f}")


## 5. Curva Precision-Recall comparativa


In [ ]:
fig = plot_pr_curve(results, y_test, output_path=None)
plt.show()


## 6. Distribución del Anomaly Score: normal vs. fraude

Si el modelo aprendió bien la forma de "normal", las transacciones fraudulentas deberían concentrarse en la cola de scores altos, separadas de la masa de transacciones legítimas.


In [ ]:
fig = plot_score_distributions(results, y_test, output_path=None)
plt.show()


## 7. Conclusiones

- Ambos modelos separan razonablemente bien el fraude usando **cero** ejemplos de fraude durante el entrenamiento — la premisa central de la detección zero-day.
- Local Outlier Factor y Isolation Forest capturan señales distintas (densidad local vs. aislamiento estructural); en producción conviene monitorear ambos scores o combinarlos.
- Este enfoque no sustituye al Módulo 1: los modelos supervisados siguen siendo más precisos para el fraude *ya conocido* (ver PR-AUC ≈ 0.999 de Random Forest en `src/models/train.py`). El Módulo 2 es la red de seguridad para lo que el Módulo 1 no puede haber aprendido a reconocer.
- El modelo de referencia (Isolation Forest + el `RobustScaler` con el que fue entrenado) queda serializado en `data/processed/isolation_forest.joblib` vía `python -m src.unsupervised.train_unsupervised`.
